# MIE1624 Fall 2025 - Indeed Job Scraper

## Features
- ✅ Automatic checkpointing every 50 jobs
- ✅ Resume from interruptions
- ✅ Enhanced salary extraction
- ✅ Support for 1000+ jobs
- ✅ Multi-search capability

---

## Quick Start

1. Run cells in order
2. Configure your search (Cell 3)
3. Run the scraper (Cell 4)
4. Analyze results (Cell 5+)

**Time for 1000 jobs:** ~3-4 hours (can run in background)

---

### Regional Considerations:

**This script uses Indeed.com (US version) by default.**

**For CANADIAN jobs:**
- Using location="Canada" on Indeed.com will return **very few results** and **limited salary data**
- For better Canadian results, you need to modify the script to use `ca.indeed.com`
- **To fix:** Open `indeed_scraper_camoufox.py` and change line ~147:
  ```python
  # Change from:
  base_url = "https://www.indeed.com/jobs"
  
  # To:
  base_url = "https://ca.indeed.com/jobs"
  ```

**For US jobs (current default):**
- Use specific locations: "New York, NY", "San Francisco, CA", or "remote"

**For Canadian jobs:**
- Use specific locations: "Toronto, ON", "Vancouver, BC", or "remote"

**For REMOTE jobs:**
- Use `LOCATION = "remote"` - works well on Indeed.com
- Returns primarily US remote jobs (good salary data)


## Step 1: System Check

In [ ]:
import sys
import platform

print("SYSTEM CHECK")
print("="*70)
print(f"OS: {platform.system()}")
print(f"Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

if sys.version_info < (3, 7):
    print("\n❌ ERROR: Python 3.7+ required")
else:
    print("\n✅ Python version OK")


## Step 2: Install Dependencies


In [ ]:
print("Installing required libraries...\n")
!pip install --quiet pandas beautifulsoup4 camoufox camoufox-captcha
print("\n✅ Installation complete!")

## Step 2.1: Verify Dependencies

In [ ]:
import importlib

print("DEPENDENCY CHECK")
print("="*70)

required = ['pandas', 'bs4', 'camoufox', 'camoufox_captcha']
all_good = True

for lib in required:
    try:
        importlib.import_module(lib)
        print(f"✅ {lib}")
    except ImportError:
        print(f"❌ {lib} - MISSING")
        all_good = False

print("="*70)
if all_good:
    print("✅ All dependencies installed!")
else:
    print("❌ Some dependencies missing - re-run Step 2")

## Step 3: Configure Search

**Customize these settings:**

- `POSITION`: Job title (e.g., "data analyst", "python developer")
- `LOCATION`: Location (e.g., "remote", "New York, NY")
- `MAX_JOBS`: How many jobs to scrape (recommended: 100-1000)

**Time estimates:**
- 100 jobs ≈ 25 minutes
- 500 jobs ≈ 2 hours
- 1000 jobs ≈ 3-4 hours

In [ ]:
# ===== CONFIGURE YOUR SEARCH =====

POSITION = "python analyst"
LOCATION = "remote"
MAX_JOBS = 10

# =================================

print("="*70)
print("SEARCH CONFIGURATION")
print("="*70)
print(f"Position: {POSITION}")
print(f"Location: {LOCATION}")
print(f"Max Jobs: {MAX_JOBS}")
print(f"\nEstimated time: {MAX_JOBS * 0.25:.0f}-{MAX_JOBS * 0.35:.0f} minutes")
print("="*70)
print("\nTIP: Start with 100 jobs, then scale up")
print("TIP: This cell can run in background - minimize and continue working")
print("="*70)

## Step 4: Run Scraper

### What happens:
1. **Checkpoint detection** - Automatically resumes if you've run this search before
2. **Browser opens** - Don't close it!
3. **Progress updates** - Every 50 jobs
4. **Auto-save** - Progress saved every 50 jobs
5. **Final CSV** - Saved when complete

### Important:
-  **Can be interrupted** - Just re-run to resume
- **Auto-saved** - Progress is never lost
- **Multiple searches** - Each search has separate checkpoints
-  **Don't close browser** - It's working!

### If something goes wrong:
- Just re-run this cell - it will resume automatically
- Check error messages for guidance
- Ask instructor for help if needed (post on Piazza is preferred or e-mail at eric.floro@mail.utoronto.ca)

In [ ]:
import subprocess
import sys
import os
from datetime import datetime

# Locate script
SCRIPT_NAME = "indeed_scraper_camoufox.py"
SCRIPT_PATH = os.path.join(os.getcwd(), SCRIPT_NAME)

if not os.path.exists(SCRIPT_PATH):
    print("="*70)
    print("❌ ERROR: Script not found!")
    print("="*70)
    print(f"\nLooking for: {SCRIPT_NAME}")
    print(f"In directory: {os.getcwd()}")
    print("\n📥 Download the script and place it in the same folder as this notebook.")
    print("\nFiles in current directory:")
    for f in os.listdir('.'):
        if f.endswith('.py'):
            print(f"  - {f}")
else:
    print("="*70)
    print("STARTING SCRAPER")
    print("="*70)
    print(f"Search: {POSITION} in {LOCATION}")
    print(f"Target: {MAX_JOBS} jobs")
    print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
    print("\n" + "="*70)
    print("SCRAPER OUTPUT")
    print("="*70 + "\n")
    
    try:
        result = subprocess.run(
            [sys.executable, SCRIPT_PATH, POSITION, LOCATION, str(MAX_JOBS)],
            capture_output=True,
            text=True
        )
        
        print(result.stdout)
        
        if result.stderr:
            # Filter out harmless warnings
            stderr_lines = result.stderr.split('\n')
            errors = [line for line in stderr_lines if line and 'charmap' not in line and 'Downloading GeoIP' not in line]
            if errors:
                print("\n" + "="*70)
                print("WARNINGS/ERRORS")
                print("="*70)
                for error in errors:
                    print(error)
        
        print("\n" + "="*70)
        print(f"End: {datetime.now().strftime('%H:%M:%S')}")
        print("="*70)
        
        if result.returncode == 0:
            print("\n✅ SCRAPING COMPLETED!")
        else:
            print(f"\n⚠️ Scraper exited with code: {result.returncode}")
            print("\n🔄 Try running this cell again - it will resume from checkpoint")
    
    except Exception as e:
        print("\n" + "="*70)
        print("❌ ERROR")
        print("="*70)
        print(f"\n{e}")
        print("\n🆘 Contact instructor with this error message")

## Step 5: Load Results

In [ ]:
import pandas as pd
import glob
import os

print("="*70)
print("LOADING RESULTS")
print("="*70)

# Find most recent CSV (not checkpoint)
csv_files = [f for f in glob.glob("*.csv") if not f.startswith('checkpoint')]

if csv_files:
    latest_file = max(csv_files, key=os.path.getctime)
    print(f"\n✅ Found: {latest_file}")
    
    df = pd.read_csv(latest_file)
    
    print(f"\n{'='*70}")
    print("DATA SUMMARY")
    print("="*70)
    print(f"Total jobs: {len(df)}")
    
    jobs_with_salary = (df['Salary'] != 'N/A').sum()
    print(f"Jobs with salary: {jobs_with_salary} ({(jobs_with_salary/len(df)*100):.1f}%)")
    print(f"Unique companies: {df['Company'].nunique()}")
    print(f"Unique locations: {df['Location'].nunique()}")
    
    print(f"\n{'='*70}")
    print("PREVIEW (First 5 rows)")
    print("="*70)
    display(df.head())
    
    print("\n✅ Data loaded successfully!")
    
else:
    print("\n❌ No results found!")
    print("\nPossible reasons:")
    print("  1. Scraper hasn't completed (check Step 4)")
    print("  2. File was deleted or moved")
    print("  3. Scraper failed to save output")
    df = None

## Step 6: Detailed Analysis

In [ ]:
if df is not None:
    print("="*70)
    print("DETAILED ANALYSIS")
    print("="*70)
    
    # Top Companies
    print("\n📊 TOP 10 COMPANIES")
    print("-"*70)
    top_companies = df['Company'].value_counts().head(10)
    for i, (company, count) in enumerate(top_companies.items(), 1):
        print(f"{i:2}. {company:40} ({count} jobs)")
    
    # Top Locations
    print("\n📍 TOP 10 LOCATIONS")
    print("-"*70)
    top_locations = df['Location'].value_counts().head(10)
    for i, (location, count) in enumerate(top_locations.items(), 1):
        print(f"{i:2}. {location:40} ({count} jobs)")
    
    # Salary Info
    jobs_with_salary = df[df['Salary'] != 'N/A']
    
    if len(jobs_with_salary) > 0:
        print("\n💰 SALARY INFORMATION")
        print("-"*70)
        print(f"Jobs with salary: {len(jobs_with_salary)}")
        print(f"\nSample salaries:")
        for i, salary in enumerate(jobs_with_salary['Salary'].head(10), 1):
            print(f"{i:2}. {salary}")
    else:
        print("\n💰 No salary information in this dataset")
        print("   (This is normal - many employers don't post salaries)")
else:
    print("❌ No data to analyze - run Step 5 first")

## Step 7: View Jobs with Salaries

In [ ]:
if df is not None:
    jobs_with_salary = df[df['Salary'] != 'N/A'][['Title', 'Company', 'Location', 'Salary']]
    
    if len(jobs_with_salary) > 0:
        print("="*70)
        print(f"JOBS WITH SALARY INFO ({len(jobs_with_salary)} jobs)")
        print("="*70)
        display(jobs_with_salary)
    else:
        print("No jobs with salary information found")
else:
    print("❌ No data loaded")


## Advanced: Run Multiple Searches



In [ ]:
# Define multiple searches
searches = [
    {"position": "data analyst", "location": "San Francisco, CA", "max_jobs": 200},
    {"position": "data scientist", "location": "remote", "max_jobs": 200},
    # Add more searches here
]

# Run each search
for i, search in enumerate(searches, 1):
    print(f"\n{'='*70}")
    print(f"SEARCH {i}/{len(searches)}")
    print("="*70)
    print(f"Position: {search['position']}")
    print(f"Location: {search['location']}")
    print(f"Max Jobs: {search['max_jobs']}")
    print("="*70)
    
    # Note: Each search will have its own checkpoint files
    # You can interrupt and resume each search independently
    
    result = subprocess.run(
        [sys.executable, SCRIPT_PATH, search['position'], search['location'], str(search['max_jobs'])],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✅ Search {i} completed!")
    else:
        print(f"\n⚠️ Search {i} had issues - check output above")

print("\n" + "="*70)
print("ALL SEARCHES COMPLETE!")
print("="*70)

---

# Troubleshooting

## Common Issues

### Issue 1: "Module not found"
**Solution:** Re-run Step 2 to install dependencies

### Issue 2: "Script not found"
**Solution:** 
1. Download `indeed_scraper_camoufox.py`
2. Place in same folder as this notebook
3. Re-run Step 4

### Issue 3: "Browser won't open"
**Solutions:**
- **Windows:** Click "Allow" when Windows Defender prompts
- **Mac:** Allow in System Preferences → Security & Privacy
- **All:** Check internet connection (browser downloads on first run)

### Issue 4: "Taking too long"
**This is normal!** Large scrapes take time:
- 100 jobs ≈ 25 minutes
- 500 jobs ≈ 2 hours
- 1000 jobs ≈ 3-4 hours

**You can:**
- Minimize browser and continue working
- Check progress every 15-30 minutes
- Interrupt and resume anytime (checkpoints save progress)

### Issue 5: "Cloudflare challenge"
**Solution:** 
- Script handles this automatically
- If manual challenge appears, complete it
- This is normal web security

### Issue 6: "No salaries found"
**This is expected** Many employers don't post salaries.
- 30-60% salary capture is excellent

---

## Tips for Best Results

1. **Start small** - Test with 50-100 jobs first
2. **Scale up** - Once working, try 500-1000 jobs
3. **Multiple searches** - Search different job titles and locations
4. **Be patient** - Large scrapes take hours
5. **Check progress** - Look for checkpoint saves every 50 jobs

---

**Questions?** Ask TA Eric Floro at eric.floro@mail.utoronto.ca